#### Open Meteo API Guide
https://open-meteo.com/en/docs

Names of data columns available for hourly can be found on the above website.

To get the exact codename to specify models, select the models and look at the weblink (should have ?models=ABC).

In [0]:
# Set up and navigate to catalog.schema workspace
catalog = "open_meteo"
schema = "datasets"
table = "dwd_api_data_raw"
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.catalog.setCurrentCatalog(catalog)
spark.catalog.setCurrentDatabase(schema)

In [0]:
# Create data table
# spark.sql("DROP TABLE IF EXISTS open_meteo.datasets.api_data")
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
    country STRING COMMENT "Country name."
    ,model STRING COMMENT "Weather model code used for forecast."
    ,forecast_date DATE COMMENT "Date of weather forecast."
    ,forecast_hour INT COMMENT "Hour of weather forecast."
    ,retrieval_timestamp BIGINT COMMENT "Unix timestamp of when the data was retrieved."
    ,temperature_2m DOUBLE COMMENT "Air temperature at 2m above ground in Celsius."
    ,apparent_temperature DOUBLE COMMENT "Perceived air temperature by taking into account the effect of wind and humidity, in Celsius"
    ,relative_humidity_2m DOUBLE COMMENT "Relative humidity at 2m above ground in %."
    ,cloud_cover DOUBLE COMMENT "Total cloud cover as an area fraction"
    ,wind_speed_10m DOUBLE COMMENT "Wind speed at 10m above ground in m/s."
    ,rain DOUBLE COMMENT "Liquid precipitation in mm."
)
USING DELTA
CLUSTER BY AUTO
COMMENT 'Raw API data from Open Meteo. Can contain multiple rows of data for the same country + model + forecast_date + forecast_hour, depending on when the data is obtained.'
''')

DataFrame[]

In [0]:
# Open Meteo API configuration variables

import requests
import pandas as pd
import time

latitude = 1.3521
longitude = 103.8198
hourly_data_columns = "temperature_2m,apparent_temperature,relative_humidity_2m,cloud_cover,wind_speed_10m,rain"
hourly_data_columns_list = hourly_data_columns.split(",")
models = [
    'ecmwf_ifs' # European Centre for Medium-Range Weather Forecasts High Resolution Forecasts 9km
    ,'ecmwf_aifs025_single' # European Centre for Medium-Range Weather Forecasts AI Forecasting System 0.25degree resolution
    ,'jma_gsm' # Japan Meteorological Agency Global Spectral Model
    ,'gem_seamless' # Global Environment Multiscale, Canadian Weather Service
    ,'icon_global'
    ,'gfs_global'
]
past_days = 3 # 0 to 92
forecast_days = 0 # 0 to 16
timezone = "Asia/Singapore"

params = {
    "latitude": latitude
    ,"longitude": longitude
    ,"hourly": hourly_data_columns
    ,"models": models
    ,"past_days": past_days
    ,"forecast_days": forecast_days
    ,"timezone": timezone
}
url = "https://api.open-meteo.com/v1/forecast"

In [0]:
response = requests.get(url, params=params)
data = response.json()
hourly_data = data.get("hourly", {}) # get the hourly data, there can be other timescales
pdf = pd.DataFrame(hourly_data) # Pandas accept a dictionary of column name to equi-length list of values, but not pyspark
pdf["time"] = pd.to_datetime(pdf["time"])
pdf.insert(0, "country", "Singapore")
pdf.insert(1, "forecast_date", pdf["time"].dt.date)
pdf.insert(2, "forecast_hour", pdf["time"].dt.hour.astype(int))
pdf.insert(3, "retrieval_timestamp", int(time.time()))
pdf = pdf.drop("time", axis=1)
sdf = spark.createDataFrame(pdf)
print(sdf.printSchema())
display(sdf)

In [0]:
# Data return from API is in wide format, stack to convert to long format
stack_parts = []
for model in models:
    stack_parts.append(f"'{model}'")
    for metric in hourly_data_columns_list:
        stack_parts.append(f"`{metric}_{model}`")
n_models = len(models)
out_cols = ", ".join(["model"] + hourly_data_columns_list)
stack_expr = f"stack({n_models}, {', '.join(stack_parts)}) as ({out_cols})"
narrow_df = sdf.select("country", "forecast_date", "forecast_hour", "retrieval_timestamp", F.expr(stack_expr))

In [0]:
narrow_df.select(
    "country",
    "model",
    "forecast_date",
    "forecast_hour",
    "retrieval_timestamp",
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "cloud_cover",
    "wind_speed_10m",
    "rain"
).write.mode("append").saveAsTable(f"{catalog}.{schema}.{table}")